# 11 · Extra analysis: our MLP with the 343 STAMP genes

The main MLP (notebooks 06–08) uses 18,040 genes. STAMP uses 344. This notebook repeats our MLP with the **343 STAMP genes without TP53**, so that the comparison with STAMP (notebook 12) can be split in two parts:

| Comparison | What changes |
|---|---|
| MLP 18,040 genes vs STAMP | Genes, network and source |
| MLP 343 genes vs STAMP | Only the network (MLP or GCN) and the source (2,860 or 625 patients) |
| MLP 18,040 genes vs MLP 343 genes | Only the genes |

The network is also much smaller: about 88,000 weights with 256 neurons, against about 4.6 million.

**Nothing new is decided here.** The same three steps are repeated with the same rules:
1. **Search (as notebook 06):** same 72 combinations, same 5 folds inside the source, same choice rule. The network is chosen again because the one of notebook 06 was chosen with 18,040 genes.
2. **Rehearsal (as notebook 07):** TCGA → METABRIC, to choose the learning rate and epochs of fine-tuning. Only strategy 2.
3. **Experiment (as notebook 08):** CPTAC and SMC, same exams and repetitions (notebook 04). Only strategies 1 (apply directly) and 2 (fine-tuning), main setting and sensitivity with a learning rate 10 times larger.

The genes come from notebook 10. SOD2 (TCGA) and TAX1BP3 (METABRIC) enter as 0 after the z-score in the cohort that does not have them.

**Output:** `results/tables/11_busqueda.csv`, `11_elegida.json`, `11_ajuste_elegido.json`, `data/experiment/red_fuente_343.keras` and `data/experiment/auc_red_343.csv`.

**Time:** not measured yet. Run it on Khipu with `khipu/notebook.slurm`. Each step saves its parts and continues from the last one if the job stops.

In [1]:
import itertools
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))
from common import armar_red, cargar, fuente, pesos_clase, zscore  # common sets Keras to run on PyTorch before keras is imported

import keras

EXP = ROOT / "data" / "experiment"
TAB = ROOT / "results" / "tables"
SEARCH = TAB / "11_busqueda.csv"
PARTS_REHEARSAL, PARTS = EXP / "partes_11_ensayo", EXP / "partes_11"
PARTS_REHEARSAL.mkdir(exist_ok=True)
PARTS.mkdir(exist_ok=True)

GENES = 343
MAX_EPOCHS, PATIENCE = 200, 10
N_LOCAL, N_EXAM, N_EXAMS, N_REPS = [10, 15, 20, 40], 40, 5, 20   # rehearsal, as notebook 07
RATES, EPOCHS = [1e-5, 1e-4, 1e-3], [10, 30, 100]                # fine-tuning grid of notebook 07
TARGETS = ["CPTAC", "SMC 2018"]


def load(name):
    x, y = cargar(name, stamp=True)
    return x.drop(columns="TP53"), y

## Step 1 · Source with the 343 genes

Same as notebook 06: TCGA and METABRIC, each one normalized with its own patients, and the same 5 folds (same rows, same order and same seed).

In [2]:
X_src, y_src, cohort_src = fuente(stamp=True)
X_src = X_src.drop(columns="TP53")
X, y = X_src.to_numpy(), y_src.to_numpy()
assert X.shape[1] == GENES
folds = list(StratifiedKFold(5, shuffle=True, random_state=2026).split(X, cohort_src + "_" + y_src.astype(str)))
print(f"{len(y):,} patients · {X.shape[1]} genes · {100 * y.mean():.1f}% mutated")

2,860 patients · 343 genes · 34.6% mutated


## Step 2 · Search (same grid and rule as notebook 06)

In [3]:
GRID = [dict(capas=c, tasa=t, lote=b, dropout=d, optimizador=o)
        for c, t, b, d, o in itertools.product([[256], [512, 128], [512, 256, 128]], [1e-4, 3e-4, 1e-3],
                                               [16, 32], [0.0, 0.3], ["adam", "sgd"])]


def key(c):
    return f"{c['capas']}|{c['tasa']}|{c['lote']}|{c['dropout']}|{c['optimizador']}"


done = set(pd.read_csv(SEARCH).combinacion) if SEARCH.exists() else set()
for c in GRID:
    if key(c) in done:
        continue
    start, rows = time.time(), []
    for f, (tr, va) in enumerate(folds):
        keras.utils.set_random_seed(2026)
        model = armar_red(c["capas"], c["dropout"], c["tasa"], c["optimizador"], GENES)
        stop = keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=PATIENCE)
        h = model.fit(X[tr], y[tr], batch_size=c["lote"], epochs=MAX_EPOCHS, validation_data=(X[va], y[va]),
                      class_weight=pesos_clase(y[tr]), callbacks=[stop], verbose=0).history
        best = int(np.argmax(h["val_auc"]))
        rows.append({"combinacion": key(c), **c, "capas": str(c["capas"]), "particion": f,
                     "auc_validacion": h["val_auc"][best], "mejor_epoca": best + 1})
    pd.DataFrame(rows).to_csv(SEARCH, mode="a", header=not SEARCH.exists(), index=False)
    print(f"{key(c)}: AUC {np.mean([r['auc_validacion'] for r in rows]):.3f} in {(time.time() - start) / 60:.1f} min")

[256]|0.0001|16|0.0|adam: AUC 0.942 in 2.3 min


[256]|0.0001|16|0.0|sgd: AUC 0.939 in 9.1 min


[256]|0.0001|16|0.3|adam: AUC 0.940 in 2.8 min


[256]|0.0001|16|0.3|sgd: AUC 0.937 in 11.0 min


[256]|0.0001|32|0.0|adam: AUC 0.941 in 1.6 min


...

[512, 256, 128]|0.001|16|0.3|sgd: AUC 0.940 in 3.0 min


[512, 256, 128]|0.001|32|0.0|adam: AUC 0.937 in 0.7 min


[512, 256, 128]|0.001|32|0.0|sgd: AUC 0.930 in 1.6 min


[512, 256, 128]|0.001|32|0.3|adam: AUC 0.939 in 0.8 min


[512, 256, 128]|0.001|32|0.3|sgd: AUC 0.938 in 2.3 min


Same rule as notebook 06: the highest mean validation AUC. If another combination is less than 0.005 below and its network is smaller, the smaller one is chosen. Then the final network is trained with the 2,860 patients and the mean number of epochs of the search.

In [4]:
search = pd.read_csv(SEARCH)
summary = (search.groupby(["combinacion", "capas", "tasa", "lote", "dropout", "optimizador"])
           .agg(auc=("auc_validacion", "mean"), de=("auc_validacion", "std"), epocas=("mejor_epoca", "mean"))
           .reset_index().sort_values("auc", ascending=False))
summary["tamaño"] = summary.capas.map(lambda s: sum(json.loads(s)))
near = summary[summary.auc >= summary.auc.max() - 0.005]
chosen = near.sort_values(["tamaño", "auc"], ascending=[True, False]).iloc[0]

arch = dict(capas=json.loads(chosen.capas), tasa=float(chosen.tasa), lote=int(chosen.lote),
            dropout=float(chosen.dropout), optimizador=chosen.optimizador, epocas=int(round(chosen.epocas)),
            auc_validacion=round(float(chosen.auc), 4))
keras.utils.set_random_seed(2026)
source = armar_red(arch["capas"], arch["dropout"], arch["tasa"], arch["optimizador"], GENES)
source.fit(X, y, batch_size=arch["lote"], epochs=arch["epocas"], class_weight=pesos_clase(y), verbose=0)
source.save(EXP / "red_fuente_343.keras")
(TAB / "11_elegida.json").write_text(json.dumps(arch, indent=2))
print(f"Weights: {source.count_params():,}")
arch

Weights: 88,321


{'capas': [256],
 'tasa': 0.0003,
 'lote': 16,
 'dropout': 0.0,
 'optimizador': 'adam',
 'epocas': 12,
 'auc_validacion': 0.9425}

## Step 3 · Rehearsal TCGA → METABRIC (as notebook 07, only fine-tuning)

A network trained only with TCGA is fine-tuned with n METABRIC patients and measured on fixed METABRIC exams. Same exams, same draws of local patients, same seeds, same grid (3 learning rates × epochs 10, 30 and 100) and same choice rule as notebook 07.

In [5]:
x_tcga, y_tcga = load("TCGA-BRCA")
X_t, y_t = zscore(x_tcga, x_tcga.index).to_numpy(), y_tcga.to_numpy()
tr, va = train_test_split(np.arange(len(y_t)), test_size=0.2, stratify=y_t, random_state=2026)
keras.utils.set_random_seed(2026)
probe = armar_red(arch["capas"], arch["dropout"], arch["tasa"], arch["optimizador"], GENES)
h = probe.fit(X_t[tr], y_t[tr], batch_size=arch["lote"], epochs=MAX_EPOCHS, validation_data=(X_t[va], y_t[va]),
              class_weight=pesos_clase(y_t[tr]), verbose=0,
              callbacks=[keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=PATIENCE)]).history
tcga_epochs = int(np.argmax(h["val_auc"])) + 1
keras.utils.set_random_seed(2026)
pretrained = armar_red(arch["capas"], arch["dropout"], arch["tasa"], arch["optimizador"], GENES)
pretrained.fit(X_t, y_t, batch_size=arch["lote"], epochs=tcga_epochs, class_weight=pesos_clase(y_t), verbose=0)
pre_weights = pretrained.get_weights()


def fine_tune(weights, rate, epochs, rep, x_loc, y_loc, x_exam, y_exam):
    """Fine-tuning with every layer free. Returns {epoch: (AUC exam, AUC local)} at each epoch in `epochs`."""
    keras.utils.set_random_seed(rep)
    model = armar_red(arch["capas"], arch["dropout"], rate, arch["optimizador"], GENES)
    model.set_weights(weights)
    out, done = {}, 0
    for ep in epochs:   # same as one run of max(epochs) measured on the way: same seed, batch and rate
        model.fit(x_loc, y_loc, batch_size=len(y_loc), epochs=ep - done, class_weight=pesos_clase(y_loc), verbose=0)
        done = ep
        out[ep] = (roc_auc_score(y_exam, model.predict(x_exam, verbose=0).ravel()),
                   roc_auc_score(y_loc, model.predict(x_loc, verbose=0).ravel()))
    return out


x_meta, y_meta = load("METABRIC")
y_m = y_meta.to_numpy()
for e in range(N_EXAMS):
    part = PARTS_REHEARSAL / f"examen{e + 1}.parquet"
    if part.exists():
        continue
    rest, exam = train_test_split(np.arange(len(y_m)), test_size=N_EXAM, stratify=y_m, random_state=2026 + e)
    z = zscore(x_meta, x_meta.index[rest]).to_numpy()
    rows = [{"examen": e + 1, "estrategia": 1, "n": 0, "auc": roc_auc_score(y_m[exam], pretrained.predict(z[exam], verbose=0).ravel())}]
    for n in N_LOCAL:
        for r in range(N_REPS):
            loc, _ = train_test_split(rest, train_size=n, stratify=y_m[rest], random_state=100000 * e + 1000 * n + r)
            for rate in RATES:
                out = fine_tune(pre_weights, rate, EPOCHS, r, z[loc], y_m[loc], z[exam], y_m[exam])
                rows += [{"examen": e + 1, "estrategia": 2, "tasa": rate, "epocas": ep, "n": n, "repeticion": r,
                          "auc": a, "auc_locales": a_loc} for ep, (a, a_loc) in out.items()]
    pd.DataFrame(rows).to_parquet(part)
    print(f"Exam {e + 1} done")

Exam 1 done


Exam 2 done


Exam 3 done


Exam 4 done


Exam 5 done


Same rule as notebook 07: the highest mean AUC on the exam (all n and exams together). If another setting is less than 0.005 below, the one with the smaller SD across repetitions. The sensitivity setting is the chosen one with a learning rate 10 times larger, as in notebook 08 (1e-5 → 1e-4).

In [6]:
rehearsal = pd.concat([pd.read_parquet(p) for p in sorted(PARTS_REHEARSAL.glob("examen*.parquet"))], ignore_index=True)
ft = rehearsal[rehearsal.estrategia == 2]
grid = (ft.groupby(["tasa", "epocas", "n", "examen"]).auc.agg(["mean", "std"])
        .groupby(["tasa", "epocas"]).mean().rename(columns={"mean": "AUC promedio", "std": "DE entre repeticiones"}))
near = grid[grid["AUC promedio"] >= grid["AUC promedio"].max() - 0.005]
rate, epochs = near["DE entre repeticiones"].idxmin()
setups = {"principal": {"tasa": float(rate), "epocas": int(epochs)},
          "sensibilidad": {"tasa": float(rate) * 10, "epocas": int(epochs)}}
(TAB / "11_ajuste_elegido.json").write_text(json.dumps({**setups, "epocas_red_tcga": tcga_epochs}, indent=2))
print("Apply directly in METABRIC:", round(rehearsal[rehearsal.estrategia == 1].auc.mean(), 3))
display(grid.round(4))
setups

Apply directly in METABRIC: 0.858


AUC promedio  DE entre repeticiones
tasa    epocas                                     
0.00001 10.0          0.8579                 0.0021
        30.0          0.8597                 0.0045
        100.0         0.8616                 0.0092
0.00010 10.0          0.8617                 0.0097
        30.0          0.8596                 0.0179
        100.0         0.8568                 0.0228
0.00100 10.0          0.8432                 0.0377
        30.0          0.8342                 0.0438
        100.0         0.8334                 0.0442

{'principal': {'tasa': 1e-05, 'epocas': 10},
 'sensibilidad': {'tasa': 0.0001, 'epocas': 10}}

## Step 4 · CPTAC and SMC (as notebook 08, strategies 1 and 2)

Same exams and repetitions as every other strategy (notebook 04). Each cohort is normalized with the patients outside the exam, without labels. Batch = all n patients, class weights and seed = repetition number, as in notebook 08.

In [7]:
exams = pd.read_csv(EXP / "examenes.csv")
reps = pd.read_csv(EXP / "repeticiones.csv")
source_weights = keras.models.load_model(EXP / "red_fuente_343.keras").get_weights()

for name in TARGETS:
    x, y = load(name)
    for e in range(1, 6):
        part = PARTS / f"{name.replace(' ', '_')}_examen{e}.parquet"
        if part.exists():
            continue
        start = time.time()
        exam_ids = exams.query("cohorte == @name and examen == @e").muestra
        z = zscore(x, x.index.difference(exam_ids))
        x_exam, y_exam = z.loc[exam_ids].to_numpy(), y.loc[exam_ids].to_numpy()
        source.set_weights(source_weights)
        rows = [{"estrategia": 1, "variante": "principal", "n": 0, "repeticion": 0,
                 "auc": roc_auc_score(y_exam, source.predict(x_exam, verbose=0).ravel())}]
        groups = reps.query("cohorte == @name and examen == @e").groupby(["n", "repeticion"]).muestra.apply(list)
        for (n, r), ids in groups.items():
            x_loc, y_loc = z.loc[ids].to_numpy(), y.loc[ids].to_numpy()
            for v, s in setups.items():
                a, a_loc = fine_tune(source_weights, s["tasa"], [s["epocas"]], r, x_loc, y_loc, x_exam, y_exam)[s["epocas"]]
                rows.append({"estrategia": 2, "variante": v, "n": n, "repeticion": r, "auc": a, "auc_locales": a_loc})
        pd.DataFrame(rows).assign(cohorte=name, examen=e).to_parquet(part)
        print(f"{name} exam {e}: {len(rows):,} networks in {(time.time() - start) / 60:.0f} min")

keys = ["cohorte", "examen", "estrategia", "variante", "n", "repeticion"]
auc = pd.concat([pd.read_parquet(p) for p in sorted(PARTS.glob("*_examen*.parquet"))], ignore_index=True)
assert auc.groupby(["cohorte", "estrategia", "variante", "n"]).size().isin([5, 500]).all(), "missing networks"
auc[keys + ["auc", "auc_locales"]].to_csv(EXP / "auc_red_343.csv", index=False)
(auc.groupby(["cohorte", "estrategia", "variante", "n", "examen"]).auc.agg(["mean", "std"])
 .groupby(["cohorte", "estrategia", "variante", "n"]).mean().round(3)
 .rename(columns={"mean": "AUC promedio", "std": "DE entre repeticiones"}))

CPTAC exam 1: 801 networks in 2 min


CPTAC exam 2: 801 networks in 2 min


CPTAC exam 3: 801 networks in 2 min


CPTAC exam 4: 801 networks in 2 min


CPTAC exam 5: 801 networks in 2 min


...

SMC 2018 exam 2: 1,001 networks in 2 min


SMC 2018 exam 3: 1,001 networks in 2 min


SMC 2018 exam 4: 1,001 networks in 2 min


SMC 2018 exam 5: 1,001 networks in 2 min


AUC promedio  DE entre repeticiones
cohorte  estrategia variante     n                                      
CPTAC    1          principal    0          0.928                    NaN
         2          principal    10         0.927                  0.001
                                 15         0.927                  0.001
                                 20         0.927                  0.001
                                 40         0.927                  0.001
                    sensibilidad 10         0.929                  0.006
                                 15         0.929                  0.006
                                 20         0.928                  0.006
                                 40         0.926                  0.005
SMC 2018 1          principal    0          0.863                    NaN
         2          principal    10         0.863                  0.002
                                 15         0.863                  0.002
                                 20         0.863                  0.002
                                 40         0.863                  0.002
                                 80         0.864                  0.001
                    sensibilidad 10         0.862                  0.008
                                 15         0.862                  0.008
                                 20         0.862                  0.008
                                 40         0.862                  0.008
                                 80         0.865                  0.005

The comparison with STAMP and with the MLP of 18,040 genes is in notebook 12.